# TF32: Precision vs Performance in Matrix Multiplication

**TensorFloat-32 (TF32)** is a math mode introduced with NVIDIA's Ampere architecture (SM80+). It is **not** a new storage format — tensors are still stored as FP32 (32-bit). Instead, TF32 is a **compute mode** used internally by Tensor Cores during matrix multiplication.

## How TF32 works

A standard FP32 number has:
- 1 sign bit, 8 exponent bits, 23 mantissa bits (total: 32 bits)

When TF32 mode is enabled, the Tensor Cores **truncate** each FP32 input to:
- 1 sign bit, 8 exponent bits, **10 mantissa bits** (same mantissa width as FP16)

The key insight: TF32 keeps the **full FP32 range** (8-bit exponent) but reduces **precision** (10-bit mantissa vs 23-bit). The accumulation is still done in full FP32, so only the inputs to each fused multiply-add are truncated.

```
FP32:  [1 sign] [8 exponent] [23 mantissa] = 32 bits  (stored & computed)
TF32:  [1 sign] [8 exponent] [10 mantissa] = 19 bits  (computed only, not stored)
FP16:  [1 sign] [5 exponent] [10 mantissa] = 16 bits
BF16:  [1 sign] [8 exponent] [7 mantissa]  = 16 bits
```

## Why TF32 exists

Tensor Cores on Ampere+ GPUs can execute TF32 matrix multiplications **much faster** than full FP32 — roughly 8x more FLOPS on A100. Since many deep learning workloads don't need all 23 bits of mantissa precision, TF32 gives a near-free speedup for training and inference.

## PyTorch control flags

PyTorch exposes two flags to control TF32 usage:

- `torch.backends.cuda.matmul.allow_tf32` — controls whether `torch.matmul` (and `@`) can use TF32 Tensor Cores. **Default: True** (since PyTorch 1.12).
- `torch.backends.cudnn.allow_tf32` — controls whether cuDNN convolutions can use TF32. **Default: True**.

Setting these to `False` forces full FP32 precision on Tensor Cores.

## What this notebook demonstrates

1. **Precision impact**: We compare FP32 matmul results (with TF32 on vs off) against an FP64 reference to quantify the precision loss from mantissa truncation.
2. **Performance impact**: We benchmark matmul across multiple matrix sizes to show the speedup TF32 provides.

In [ ]:
!pip install -q torch triton

In [ ]:
import torch
import triton
import matplotlib.pyplot as plt
import numpy as np

assert torch.cuda.is_available(), "CUDA GPU required"
major, _ = torch.cuda.get_device_capability()
assert major >= 8, f"TF32 requires Ampere (SM80+), got SM{major}0"

print(f"GPU: {torch.cuda.get_device_name()}")
print(f"Compute capability: {torch.cuda.get_device_capability()}")
print(f"PyTorch version: {torch.__version__}")

## Part 1: Precision Comparison

We compute `C = A @ B` three ways:
1. **FP64 reference** — double-precision matmul (highest accuracy, our ground truth)
2. **FP32 with TF32 disabled** — full single-precision on Tensor Cores
3. **FP32 with TF32 enabled** — Tensor Cores truncate inputs to 10-bit mantissa

We then measure the error of both FP32 variants against the FP64 reference.

In [ ]:
def measure_precision(M, K, N, seed=None, return_errors=False):
    """Compare matmul precision: TF32-enabled FP32 vs TF32-disabled FP32 vs FP64 reference."""
    if seed is not None:
        torch.manual_seed(seed)

    # Generate random matrices in FP32
    A_fp32 = torch.randn(M, K, device="cuda", dtype=torch.float32)
    B_fp32 = torch.randn(K, N, device="cuda", dtype=torch.float32)

    # FP64 reference (ground truth)
    A_fp64 = A_fp32.double()
    B_fp64 = B_fp32.double()
    C_ref = A_fp64 @ B_fp64

    # FP32 with TF32 disabled (full precision)
    torch.backends.cuda.matmul.allow_tf32 = False
    C_fp32 = A_fp32 @ B_fp32

    # FP32 with TF32 enabled (truncated mantissa)
    torch.backends.cuda.matmul.allow_tf32 = True
    C_tf32 = A_fp32 @ B_fp32

    # Compute errors in FP64 to avoid contamination from casting the reference down
    fp32_abs_err = (C_fp32.double() - C_ref).abs()
    tf32_abs_err = (C_tf32.double() - C_ref).abs()

    # Relative error (avoid division by zero)
    C_ref_abs = C_ref.abs().clamp(min=1e-8)
    fp32_rel_err = fp32_abs_err / C_ref_abs
    tf32_rel_err = tf32_abs_err / C_ref_abs

    stats = {
        "fp32_max_abs": fp32_abs_err.max().item(),
        "tf32_max_abs": tf32_abs_err.max().item(),
        "fp32_mean_abs": fp32_abs_err.mean().item(),
        "tf32_mean_abs": tf32_abs_err.mean().item(),
        "fp32_max_rel": fp32_rel_err.max().item(),
        "tf32_max_rel": tf32_rel_err.max().item(),
        "fp32_mean_rel": fp32_rel_err.mean().item(),
        "tf32_mean_rel": tf32_rel_err.mean().item(),
    }

    if return_errors:
        return stats, fp32_abs_err.float(), tf32_abs_err.float()
    return stats

In [ ]:
sizes = [256, 512, 1024, 2048, 4096]

print(f"{'Size':>10} | {'FP32 Mean Abs Err':>18} | {'TF32 Mean Abs Err':>18} | {'FP32 Mean Rel Err':>18} | {'TF32 Mean Rel Err':>18}")
print("-" * 95)

all_results = []
for i, N in enumerate(sizes):
    res = measure_precision(N, N, N, seed=42 + i)
    all_results.append(res)
    print(f"{N:>10} | {res['fp32_mean_abs']:>18.6e} | {res['tf32_mean_abs']:>18.6e} | {res['fp32_mean_rel']:>18.6e} | {res['tf32_mean_rel']:>18.6e}")

### Visualizing the Precision Gap

The bar chart below shows mean relative error for FP32 (TF32 off) vs FP32 (TF32 on) at each matrix size. The error grows with matrix size because larger reductions accumulate more rounding errors from the truncated mantissa.

In [ ]:
x = np.arange(len(sizes))
width = 0.35

fp32_rel = [r["fp32_mean_rel"] for r in all_results]
tf32_rel = [r["tf32_mean_rel"] for r in all_results]

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width / 2, fp32_rel, width, label="FP32 (TF32 off)", color="steelblue", edgecolor="black")
bars2 = ax.bar(x + width / 2, tf32_rel, width, label="FP32 (TF32 on)", color="indianred", edgecolor="black")

# Annotate ratio
for i in range(len(sizes)):
    ratio = tf32_rel[i] / fp32_rel[i] if fp32_rel[i] > 0 else 0
    top = max(fp32_rel[i], tf32_rel[i])
    ax.text(x[i], top * 1.05, f"{ratio:.1f}x", ha="center", va="bottom", fontsize=9, fontweight="bold")

ax.set_xlabel("Matrix Size (N x N)")
ax.set_ylabel("Mean Relative Error vs FP64")
ax.set_title("Precision Loss: TF32 vs Full FP32 (relative to FP64 reference)")
ax.set_xticks(x)
ax.set_xticklabels([str(s) for s in sizes])
ax.legend()
ax.ticklabel_format(axis="y", style="scientific", scilimits=(0, 0))

plt.tight_layout()
plt.show()

### Error Distribution

Let's look at the element-wise error distribution for a single large matmul to understand how TF32 errors are distributed compared to FP32.

In [ ]:
N = 2048
res, fp32_abs_err, tf32_abs_err = measure_precision(N, N, N, seed=123, return_errors=True)

fp32_errs = fp32_abs_err.cpu().numpy().flatten()
tf32_errs = tf32_abs_err.cpu().numpy().flatten()

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(fp32_errs, bins=100, alpha=0.7, label="FP32 (TF32 off)", color="steelblue", density=True)
ax.hist(tf32_errs, bins=100, alpha=0.7, label="FP32 (TF32 on)", color="indianred", density=True)
ax.set_xlabel("Absolute Error vs FP64 Reference")
ax.set_ylabel("Density")
ax.set_title(f"Error Distribution for {N}x{N} Matmul")
ax.legend()
plt.tight_layout()
plt.show()

print(f"FP32 (TF32 off) — mean: {fp32_errs.mean():.6e}, max: {fp32_errs.max():.6e}")
print(f"FP32 (TF32 on)  — mean: {tf32_errs.mean():.6e}, max: {tf32_errs.max():.6e}")

## Part 2: Performance Benchmarking

Now we benchmark `torch.matmul` with TF32 enabled vs disabled across a range of matrix sizes using `triton.testing.do_bench`. On Ampere+ GPUs, TF32 should be significantly faster because Tensor Cores process the reduced-precision inputs at higher throughput.

In [ ]:
bench_sizes = [256, 512, 1024, 2048, 4096, 8192]

bench_results = []

print(f"{'Size':>8} | {'FP32 (TF32 off) ms':>19} | {'FP32 (TF32 on) ms':>18} | {'Speedup':>8}")
print("-" * 62)

for N in bench_sizes:
    A = torch.randn(N, N, device="cuda", dtype=torch.float32)
    B = torch.randn(N, N, device="cuda", dtype=torch.float32)

    # Benchmark with TF32 disabled
    torch.backends.cuda.matmul.allow_tf32 = False
    ms_fp32 = triton.testing.do_bench(lambda: A @ B, warmup=25, rep=100)

    # Benchmark with TF32 enabled
    torch.backends.cuda.matmul.allow_tf32 = True
    ms_tf32 = triton.testing.do_bench(lambda: A @ B, warmup=25, rep=100)

    speedup = ms_fp32 / ms_tf32
    bench_results.append((N, ms_fp32, ms_tf32, speedup))
    print(f"{N:>8} | {ms_fp32:>19.4f} | {ms_tf32:>18.4f} | {speedup:>7.2f}x")

In [ ]:
labels = [str(s) for s in bench_sizes]
fp32_times = [r[1] for r in bench_results]
tf32_times = [r[2] for r in bench_results]
speedups = [r[3] for r in bench_results]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width / 2, fp32_times, width, label="FP32 (TF32 off)", color="steelblue", edgecolor="black")
bars2 = ax.bar(x + width / 2, tf32_times, width, label="FP32 (TF32 on)", color="seagreen", edgecolor="black")

for i, (b1, b2, s) in enumerate(zip(bars1, bars2, speedups)):
    top = max(b1.get_height(), b2.get_height())
    ax.text(x[i], top * 1.02, f"{s:.2f}x", ha="center", va="bottom", fontsize=9, fontweight="bold")

ax.set_xlabel("Matrix Size (N x N)")
ax.set_ylabel("Time (ms)")
ax.set_title("Matmul Performance: TF32 On vs Off")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()

plt.tight_layout()
plt.show()

## TFLOPS Comparison

To better understand the throughput difference, let's compute TFLOPS (tera floating-point operations per second) for each configuration. A matmul of two NxN matrices requires `2 * N^3` FLOPs.

In [ ]:
print(f"{'Size':>8} | {'FP32 TFLOPS':>12} | {'TF32 TFLOPS':>12} | {'Speedup':>8}")
print("-" * 50)

for N, ms_fp32, ms_tf32, speedup in bench_results:
    flops = 2 * N**3
    tflops_fp32 = flops / (ms_fp32 * 1e-3) / 1e12
    tflops_tf32 = flops / (ms_tf32 * 1e-3) / 1e12
    print(f"{N:>8} | {tflops_fp32:>12.2f} | {tflops_tf32:>12.2f} | {speedup:>7.2f}x")

## Analysis

**Precision:**
- TF32 introduces measurably higher error than full FP32, typically an order of magnitude more relative error vs the FP64 reference. This is expected: truncating from 23 to 10 mantissa bits loses ~13 bits of precision per input element, and these errors accumulate across the K-dimension reduction.
- The absolute errors grow with matrix size because each output element is a dot product of longer vectors, accumulating more truncation errors.
- For most deep learning workloads, this level of error is acceptable — SGD is inherently noisy, and model accuracy is rarely sensitive to the last 13 bits of mantissa.

**Performance:**
- TF32 provides significant speedups on Tensor Core-equipped GPUs, especially for large matrices where the computation is Tensor Core-bound rather than memory-bound.
- Small matrices may show less speedup because launch overhead and memory access dominate.
- The speedup comes from Tensor Cores processing TF32 inputs at higher throughput — the TFLOPS table above shows the concrete difference on this GPU.

**Practical guidance:**
- For training neural networks: leave TF32 enabled (the default). The precision loss is negligible for convergence.
- For scientific computing or applications requiring FP32 exactness: disable TF32 with `torch.backends.cuda.matmul.allow_tf32 = False`.
- For inference: TF32 is almost always safe and provides free throughput.
- Remember: TF32 only affects operations that use Tensor Cores (matmul, conv). Element-wise ops always use full FP32.